In [8]:
import os
import json
import requests
from langchain_openai import ChatOpenAI

# 1. Config
MCP_SERVER_URL = "http://localhost:8000/mcp"
os.environ["OPENAI_API_KEY"]  # make sure your API key is set in env

# 2. Setup LLM
llm = ChatOpenAI(model="gpt-4o-mini")


In [2]:

# 3. Define helper to call MCP tools
def call_mcp_tool(tool_name: str, payload: dict):
    """Call MCP tool by name with JSON payload"""
    url = f"{MCP_SERVER_URL}/tools/{tool_name}"
    resp = requests.post(url, json=payload)
    return resp.json()


In [22]:
import asyncio
from langgraph.prebuilt import create_react_agent
from langchain_mcp_adapters.client import MultiServerMCPClient

async def get_mcp_list():
    """Call MCP tool by name with JSON payload"""
    servers = {
        "orian": {
            "url": "http://localhost:8000/mcp",
            "transport": "streamable_http"
        }
    }

    client = MultiServerMCPClient(servers)
    tools = await client.get_tools()   # this internally calls session/start → tools/list

    return tools

# 4. Example usage
tools = await get_mcp_list()
print("Available tools:", tools)

Available tools: [StructuredTool(name='get_orian_status', description='\n    Mirrors GET /admin/get_status as an MCP tool.\n    ', args_schema={'properties': {}, 'title': 'get_orian_statusArguments', 'type': 'object'}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x00000163638C0160>), StructuredTool(name='get_weather', description='\n    Mirrors GET /admin/get_weather as an MCP tool.\n    ', args_schema={'properties': {}, 'title': 'get_weatherArguments', 'type': 'object'}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x00000163638C2B00>)]


C:\Users\madhu\AppData\Local\Temp\ipykernel_23724\1596288425.py:20: RuntimeWarning: coroutine 'get_mcp_list' was never awaited
  tools = await get_mcp_list()


In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import initialize_agent, AgentType

# LLM
llm = ChatOpenAI(model="gpt-4o-mini")

# Suppose you already have your tools as `mcp_tools = [tool1, tool2]`
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.OPENAI_MULTI_FUNCTIONS,  # lets LLM decide which tool
    verbose=True
)

query = "You are a helpful assistant that can use various tools. Please help me with the following task: Find the current weather in New York City and get the orian status."

res = await agent.ainvoke({"input": query})
print(res)




> Entering new AgentExecutor chain...

Invoking: `get_weather` with `{'location': 'New York City'}`



Invoking: `get_orian_status` with `{}`


{
  "status": "running"
}{
  "weather": "sunny"
}The current weather in New York City is sunny, and the Orian status is running.

> Finished chain.
{'input': 'You are a helpful assistant that can use various tools. Please help me with the following task: Find the current weather in New York City and get the orian status.', 'output': 'The current weather in New York City is sunny, and the Orian status is running.'}


In [ ]:
# Create workflow from query using proper serialization
def create_workflow(query: str):
    # Create context definition
    requester_context = ContextDef(
        key="REQUESTER",
        description="Person who requested the workflow",
        dataType="STRING"
    ).model_dump()  # Convert to dict

    # Parse the tasks from query
    if "onboarding" in query.lower() and "offboarding" in query.lower():
        # Create onboarding outcomes
        onboarding_outcomes = [
            OutcomeDef(
                name="COMPLETED",
                description="Task completed successfully",
                nextTask="OFFBOARDING"
            ).model_dump(),
            OutcomeDef(
                name="REJECTED",
                description="Task was rejected",
                nextTask=None
            ).model_dump()
        ]

        # Create offboarding outcomes
        offboarding_outcomes = [
            OutcomeDef(
                name="COMPLETED",
                description="Task completed successfully",
                nextTask=None
            ).model_dump(),
            OutcomeDef(
                name="REJECTED",
                description="Task was rejected",
                nextTask=None
            ).model_dump()
        ]

        # Create tasks
        tasks = [
            TaskDef(
                taskName="ONBOARDING",
                taskDetails="Process new employee onboarding",
                taskType="MANUAL",
                assignmentType="USER",
                sequence=1,
                outcomes=onboarding_outcomes
            ).model_dump(),
            TaskDef(
                taskName="OFFBOARDING",
                taskDetails="Process employee offboarding",
                taskType="MANUAL",
                assignmentType="USER",
                sequence=2,
                outcomes=offboarding_outcomes
            ).model_dump()
        ]

        # Create the complete workflow configuration
        workflow_config = ConfigureWorkflowInput(
            workflowName="Employee Processing Flow",
            description="Handle employee onboarding and offboarding process",
            app="Agent Orc",
            contexts=[requester_context],
            tasks=tasks,
            startInstance=False,
            initialContext={"REQUESTER": "user@example.com"}
        )

        # Call the configuration function with the model data
        result = configure_workflow_func(**workflow_config.model_dump())
        return result

# Test the workflow creation
query = "create a workflow with 2 tasks, onboarding and offboarding. Once onboarding completed offboard should be the next task"
result = create_workflow(query)
print("Workflow Configuration Result:")
print(json.dumps(result, indent=2))

In [24]:
# Function to generate workflow configuration using LLM
def generate_workflow_from_query(query: str):
    # Create a prompt that explains the required structure
    structured_prompt = f"""
    Based on this user request: "{query}"
    
    Generate a workflow configuration that exactly matches these Pydantic models:

    class OutcomeDef(BaseModel):
        name: str
        description: Optional[str]
        nextTask: Optional[str]

    class ContextDef(BaseModel):
        key: str
        description: str
        dataType: str

    class TaskDef(BaseModel):
        taskName: str
        taskDetails: str
        taskType: str
        assignmentType: str
        sequence: int
        outcomes: List[OutcomeDef]

    class ConfigureWorkflowInput(BaseModel):
        workflowName: str
        description: str
        app: str
        contexts: List[ContextDef]
        tasks: List[TaskDef]
        startInstance: bool
        initialContext: Optional[Dict[str, Any]]

    Rules:
    1. Response must be ONLY the valid JSON configuration
    2. taskType should be "MANUAL"
    3. assignmentType should be "USER"
    4. Include REQUESTER in contexts
    5. Connect tasks using nextTask in outcomes
    6. Task sequences should start from 1
    7. Each task must have at least one outcome
    """

    # Get the configuration from LLM
    result = llm.invoke(structured_prompt)
    
    try:
        # Parse the LLM response to get just the JSON
        config_dict = json.loads(result.content)
        
        # Validate using Pydantic model
        config = ConfigureWorkflowInput(**config_dict)
        
        # Call the workflow configuration function
        result = configure_workflow_func(**config.model_dump())
        return {"status": "success", "result": result, "config": config.model_dump()}
    except Exception as e:
        return {"status": "error", "error": str(e)}

# Test with a sample query
query = "create a workflow with 2 tasks, onboarding and offboarding. Once onboarding completed offboard should be the next task"
result = generate_workflow_from_query(query)
print("Generated Workflow Configuration:")
print(json.dumps(result, indent=2))

Generated Workflow Configuration:
{
  "status": "error",
  "error": "Expecting value: line 1 column 1 (char 0)"
}


In [ ]:
# Example of expected workflow configuration
example_config = {
    "workflowName": "Employee Processing",
    "description": "Handle employee onboarding and offboarding process",
    "app": "Agent Orc",
    "contexts": [
        {
            "key": "REQUESTER",
            "description": "Person who initiated the process",
            "dataType": "STRING"
        }
    ],
    "tasks": [
        {
            "taskName": "ONBOARDING",
            "taskDetails": "Complete employee onboarding process",
            "taskType": "MANUAL",
            "assignmentType": "USER",
            "sequence": 1,
            "outcomes": [
                {
                    "name": "COMPLETED",
                    "description": "Onboarding process completed successfully",
                    "nextTask": "OFFBOARDING"
                },
                {
                    "name": "REJECTED",
                    "description": "Onboarding process was rejected",
                    "nextTask": None
                }
            ]
        },
        {
            "taskName": "OFFBOARDING",
            "taskDetails": "Process employee offboarding",
            "taskType": "MANUAL",
            "assignmentType": "USER",
            "sequence": 2,
            "outcomes": [
                {
                    "name": "COMPLETED",
                    "description": "Offboarding process completed",
                    "nextTask": None
                },
                {
                    "name": "REJECTED",
                    "description": "Offboarding process was rejected",
                    "nextTask": None
                }
            ]
        }
    ],
    "startInstance": False,
    "initialContext": {
        "REQUESTER": "user@example.com"
    }
}

# Test the configuration
result = agent.run("Configure a workflow using this exact structure: " + json.dumps(example_config, indent=2))
print(result)